# Agent Evaluation Framework: Outcome & Trajectory Metrics with Mistral AI

<a href="https://colab.research.google.com/github/mistralai/cookbook/blob/main/mistral/evaluation/agent_evaluation_framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Evaluating LLM-based agents requires going beyond simple output accuracy. This notebook introduces a **comprehensive evaluation framework** that measures both:

- **Outcome-level metrics**: Did the agent complete the task correctly?
- **Trajectory-level metrics**: Did the agent make good decisions along the way?

We use Mistral's structured outputs and LLM-as-Judge pattern to automate evaluation.

## Installation

In [ ]:
!pip install mistralai==1.5.0 pydantic==2.10.6 matplotlib==3.9.4 pandas==2.2.3 numpy==1.26.4

## Setup

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Optional, Dict, Any
from enum import Enum
from pydantic import BaseModel, Field
from getpass import getpass
from mistralai import Mistral

api_key = getpass("Type your API Key")
client = Mistral(api_key=api_key)

## Step 1: Define Evaluation Metrics

We define two categories of metrics, each with structured Pydantic models.

### Outcome-Level Metrics
Evaluate the **final result** of the agent's work:
- **Task completion**: Did the agent produce the expected output?
- **Answer correctness**: Is the answer factually correct?
- **Format compliance**: Does the output match the required format?

### Trajectory-Level Metrics
Evaluate the **process** the agent followed:
- **Tool selection**: Did the agent choose appropriate tools?
- **Reasoning quality**: Was the chain of thought logical and efficient?
- **Error recovery**: Did the agent handle errors gracefully?

In [ ]:
class Score(str, Enum):
    EXCELLENT = "4"
    GOOD = "3"
    FAIR = "2"
    POOR = "1"

# --- Outcome Metrics ---
class TaskCompletionScore(BaseModel):
    """Evaluate whether the agent completed the assigned task."""
    score: Score = Field(description="4=fully completed, 3=mostly completed, 2=partially, 1=failed")
    reasoning: str = Field(description="Explanation of the score")
    expected_vs_actual: str = Field(description="Comparison of expected output with actual output")

class AnswerCorrectnessScore(BaseModel):
    """Evaluate factual correctness of the agent's answer."""
    score: Score = Field(description="4=fully correct, 3=minor errors, 2=significant errors, 1=incorrect")
    reasoning: str = Field(description="Explanation with specific factual checks")
    errors_found: List[str] = Field(default_factory=list, description="List of factual errors if any")

class FormatComplianceScore(BaseModel):
    """Evaluate whether the output matches the required format."""
    score: Score = Field(description="4=perfect format, 3=minor deviations, 2=major deviations, 1=wrong format")
    reasoning: str = Field(description="Details about format compliance")

# --- Trajectory Metrics ---
class ToolSelectionScore(BaseModel):
    """Evaluate the appropriateness of tools chosen by the agent."""
    score: Score = Field(description="4=optimal tools, 3=good choices, 2=suboptimal, 1=wrong tools")
    reasoning: str = Field(description="Analysis of tool selection decisions")
    optimal_tools: List[str] = Field(description="What the optimal tool sequence would have been")

class ReasoningQualityScore(BaseModel):
    """Evaluate the quality of the agent's reasoning chain."""
    score: Score = Field(description="4=excellent reasoning, 3=good, 2=flawed logic, 1=poor reasoning")
    reasoning: str = Field(description="Analysis of the reasoning chain")
    logical_gaps: List[str] = Field(default_factory=list, description="Identified gaps in reasoning")

class ErrorRecoveryScore(BaseModel):
    """Evaluate how the agent handled errors during execution."""
    score: Score = Field(description="4=graceful recovery, 3=adequate, 2=partial recovery, 1=no recovery")
    reasoning: str = Field(description="Analysis of error handling")
    errors_encountered: List[str] = Field(default_factory=list, description="Errors that occurred")

## Step 2: Test Scenarios

We define realistic agent scenarios with expected behaviors and ground truth.

In [ ]:
test_scenarios = [
    {
        "id": "scenario_1",
        "name": "Information Extraction with Tool Use",
        "description": "Agent must extract structured data from a medical report using search and extraction tools",
        "input": "Extract patient demographics, diagnosis, and prescribed medications from this medical note: 'Patient John Doe, 45M, presented with persistent cough and fever. Diagnosed with community-acquired pneumonia. Prescribed Amoxicillin 500mg TID for 7 days and Ibuprofen 400mg PRN.'",
        "expected_output": {
            "patient_name": "John Doe",
            "age": 45,
            "gender": "Male",
            "diagnosis": "community-acquired pneumonia",
            "medications": [
                {"name": "Amoxicillin", "dose": "500mg", "frequency": "TID", "duration": "7 days"},
                {"name": "Ibuprofen", "dose": "400mg", "frequency": "PRN"}
            ]
        },
        "available_tools": ["text_extraction", "json_formatter", "medical_lookup"],
        "optimal_tool_sequence": ["text_extraction", "json_formatter"],
        "evaluation_focus": "extraction_accuracy"
    },
    {
        "id": "scenario_2",
        "name": "Multi-Step Reasoning with Error Handling",
        "description": "Agent must solve a math problem requiring multiple steps, including handling a misleading detail",
        "input": "A store sells apples for $2 each and oranges for $3 each. A customer buys 5 apples and some oranges. The total bill before a 10% discount is $25. Note: the store is located on 5th Avenue (irrelevant). How many oranges did the customer buy?",
        "expected_output": "5 oranges",
        "reasoning_steps": [
            "Identify relevant information (prices and total)",
            "Filter out irrelevant detail (store location)",
            "Calculate: 5 apples * $2 = $10",
            "Remaining before discount: $25 - $10 = $15",
            "Oranges: $15 / $3 = 5"
        ],
        "available_tools": ["calculator", "text_parser"],
        "optimal_tool_sequence": ["text_parser", "calculator"],
        "evaluation_focus": "reasoning"
    },
    {
        "id": "scenario_3",
        "name": "Code Generation with Specification Compliance",
        "description": "Agent must generate code that meets specific requirements",
        "input": "Write a Python function `merge_sorted_lists(list1, list2)` that merges two sorted lists into a single sorted list in O(n+m) time. Include type hints, docstring, and handle edge cases (empty lists, None inputs).",
        "expected_output": "A function with correct merge logic, type hints, docstring, and edge case handling",
        "format_requirements": "Python function with type hints, docstring, edge case handling",
        "available_tools": ["code_generator", "code_validator", "documentation_generator"],
        "optimal_tool_sequence": ["code_generator", "code_validator"],
        "evaluation_focus": "code_quality"
    }
]

print(f"Defined {len(test_scenarios)} test scenarios:")
for s in test_scenarios:
    print(f"  - {s['name']}")

## Step 3: Simulated Agent Execution

We simulate agent execution using Mistral to generate realistic agent traces (action sequences with tool calls).

### Considerations:
- In production, traces would come from real agent frameworks (e.g., Mistral Agents API)
- Simulation allows this notebook to be self-contained and runnable without external dependencies

In [ ]:
class AgentTrace(BaseModel):
    """A record of an agent's execution trajectory."""
    steps: List[Dict[str, Any]] = Field(description="Sequence of (action, tool_used, output) tuples")
    final_output: str = Field(description="The agent's final response")
    errors: List[str] = Field(default_factory=list, description="Errors encountered during execution")

def simulate_agent(scenario: dict) -> AgentTrace:
    """Simulate an agent executing a scenario using Mistral Large."""
    
    prompt = f"""You are simulating an AI agent executing a task. Generate a realistic execution trace.

Task: {scenario['input']}
Available tools: {scenario['available_tools']}

Generate a JSON response with:
1. "steps": array of objects with "action" (what the agent decided), "tool_used" (which tool, or "none"), "output" (result of the step)
2. "final_output": the agent's final answer
3. "errors": any errors encountered (can be empty)

Be realistic: sometimes make suboptimal choices or encounter minor issues."""

    response = client.chat.complete(
        model="mistral-large-latest",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    data = json.loads(response.choices[0].message.content)
    return AgentTrace(**data)

# Run simulations
agent_traces = {}
for scenario in test_scenarios:
    print(f"Simulating: {scenario['name']}...")
    trace = simulate_agent(scenario)
    agent_traces[scenario["id"]] = trace
    print(f"  Steps: {len(trace.steps)} | Errors: {len(trace.errors)}")

## Step 4: LLM-as-Judge Evaluation

We use Mistral Large as a judge to score each metric using structured outputs.

### Considerations:
- **Structured outputs** ensure consistent, parseable evaluation scores
- **Separate calls per metric** allow fine-grained analysis
- **Judge temperature = 0** for reproducibility

In [ ]:
def evaluate_with_judge(scenario: dict, trace: AgentTrace, metric_model: type, metric_name: str) -> BaseModel:
    """Use Mistral Large as a judge to evaluate a specific metric."""
    
    eval_prompt = f"""You are an expert evaluator assessing an AI agent's performance.

## Scenario
Task: {scenario['input']}
Expected output: {json.dumps(scenario.get('expected_output', ''), default=str)}

## Agent Execution Trace
Steps taken:
{json.dumps(trace.steps, indent=2, default=str)}

Final output: {trace.final_output}
Errors encountered: {trace.errors}

## Evaluation Metric: {metric_name}
Score the agent's performance on this specific metric.
Available tools were: {scenario.get('available_tools', [])}
Optimal tool sequence: {scenario.get('optimal_tool_sequence', [])}"""

    response = client.chat.complete(
        model="mistral-large-latest",
        messages=[{"role": "user", "content": eval_prompt}],
        response_format={"type": "json_object"},
        temperature=0
    )
    
    data = json.loads(response.choices[0].message.content)
    return metric_model.model_validate(data)

In [ ]:
evaluation_results = {}

outcome_metrics = [
    (TaskCompletionScore, "task_completion"),
    (AnswerCorrectnessScore, "answer_correctness"),
    (FormatComplianceScore, "format_compliance")
]
trajectory_metrics = [
    (ToolSelectionScore, "tool_selection"),
    (ReasoningQualityScore, "reasoning_quality"),
    (ErrorRecoveryScore, "error_recovery")
]

for scenario in test_scenarios:
    scenario_id = scenario["id"]
    trace = agent_traces[scenario_id]
    evaluation_results[scenario_id] = {"outcome": {}, "trajectory": {}}
    
    print(f"\nEvaluating: {scenario['name']}")
    
    # Outcome metrics
    for metric_model, metric_name in outcome_metrics:
        result = evaluate_with_judge(scenario, trace, metric_model, metric_name)
        evaluation_results[scenario_id]["outcome"][metric_name] = result
        print(f"  [OUTCOME] {metric_name}: {result.score.value}/4 - {result.reasoning[:60]}...")
    
    # Trajectory metrics
    for metric_model, metric_name in trajectory_metrics:
        result = evaluate_with_judge(scenario, trace, metric_model, metric_name)
        evaluation_results[scenario_id]["trajectory"][metric_name] = result
        print(f"  [TRAJECTORY] {metric_name}: {result.score.value}/4 - {result.reasoning[:60]}...")

## Step 5: Structured Evaluation Report

In [ ]:
# Build summary DataFrame
rows = []
for scenario in test_scenarios:
    sid = scenario["id"]
    evals = evaluation_results[sid]
    row = {"scenario": scenario["name"]}
    
    for category in ["outcome", "trajectory"]:
        for metric_name, result in evals[category].items():
            row[f"{category}_{metric_name}"] = int(result.score.value)
    
    rows.append(row)

report_df = pd.DataFrame(rows)

# Calculate composite scores
outcome_cols = [c for c in report_df.columns if c.startswith("outcome_")]
trajectory_cols = [c for c in report_df.columns if c.startswith("trajectory_")]

report_df["outcome_avg"] = report_df[outcome_cols].mean(axis=1).round(2)
report_df["trajectory_avg"] = report_df[trajectory_cols].mean(axis=1).round(2)
report_df["overall_score"] = ((report_df["outcome_avg"] + report_df["trajectory_avg"]) / 2).round(2)

print("=== AGENT EVALUATION REPORT ===\n")
print(report_df.to_string(index=False))

## Step 6: Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Bar chart: scores by scenario
metrics = outcome_cols + trajectory_cols
x = np.arange(len(report_df))
width = 0.12
for i, metric in enumerate(metrics):
    short_name = metric.split("_", 1)[1]
    axes[0].bar(x + i * width, report_df[metric], width, label=short_name)

axes[0].set_xlabel("Scenario")
axes[0].set_ylabel("Score (1-4)")
axes[0].set_title("Detailed Metrics by Scenario")
axes[0].set_xticks(x + width * len(metrics) / 2)
axes[0].set_xticklabels([s["name"][:20] + "..." for s in test_scenarios], rotation=15)
axes[0].legend(fontsize=7, ncol=2)
axes[0].set_ylim(0, 5)

# 2. Outcome vs Trajectory comparison
for idx, row in report_df.iterrows():
    values = [row["outcome_avg"], row["trajectory_avg"]]
    axes[1].barh(idx, values[0], color="#4ECDC4", alpha=0.7, label="Outcome" if idx == 0 else "")
    axes[1].barh(idx, values[1], left=values[0], color="#FF6B6B", alpha=0.7, label="Trajectory" if idx == 0 else "")

axes[1].set_yticks(range(len(report_df)))
axes[1].set_yticklabels([s["name"][:25] for s in test_scenarios])
axes[1].set_xlabel("Cumulative Score")
axes[1].set_title("Outcome vs Trajectory Scores")
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 7: Detailed Failure Analysis

We analyze the lowest-scoring metrics to identify patterns and improvement areas.

In [ ]:
print("=== FAILURE ANALYSIS ===\n")
for scenario in test_scenarios:
    sid = scenario["id"]
    evals = evaluation_results[sid]
    print(f"\n--- {scenario['name']} ---")
    
    all_scores = {}
    for category in ["outcome", "trajectory"]:
        for metric_name, result in evals[category].items():
            score = int(result.score.value)
            all_scores[f"{category}/{metric_name}"] = score
            if score <= 2:
                print(f"  WARNING [{metric_name}]: Score {score}/4")
                print(f"    Reason: {result.reasoning}")
                if hasattr(result, 'errors_found') and result.errors_found:
                    print(f"    Errors: {result.errors_found}")
                if hasattr(result, 'logical_gaps') and result.logical_gaps:
                    print(f"    Gaps: {result.logical_gaps}")
    
    if all(v >= 3 for v in all_scores.values()):
        print(f"  OK - All metrics >= 3/4. Agent performed well on this scenario.")

## Summary

This framework provides a systematic approach to agent evaluation with two complementary dimensions:

| Dimension | Metrics | Purpose |
|-----------|---------|----------|
| **Outcome** | Task completion, Answer correctness, Format compliance | Was the final result correct? |
| **Trajectory** | Tool selection, Reasoning quality, Error recovery | Was the process sound? |

### Key insight
An agent can produce the right answer via a wrong process (lucky outcome, poor trajectory) or follow perfect reasoning but make a formatting error (good trajectory, poor outcome). **Both dimensions are essential** for production-ready agent evaluation.

### Considerations:
- **Extending metrics**: Add domain-specific metrics by creating new Pydantic models and judge prompts
- **Automated CI/CD**: Integrate this framework into regression testing for agent deployments
- **Benchmark datasets**: Build larger, domain-specific scenario sets for robust evaluation
- **Judge calibration**: Periodically validate the LLM judge against human evaluators
- **Multi-model comparison**: Run the same scenarios with different Mistral models to compare agent capabilities